> Projeto Desenvolve <br>
Programação Intermediária com Python <br>
Profa. Camila Laranjeira (mila@projetodesenvolve.com.br) <br>

# 3.14 - ORM

## Exercícios

#### Q1. Conhecendo os dados
Baixe o seguinte csv onde iremos trabalhar. Ele contém informações sobre salários de profissionais de dados de uma empresa hipotética entre 2009 e 2016
* https://github.com/camilalaranjeira/python-intermediario/blob/main/salaries.csv

Suas colunas, descritas na [página do Kaggle que contém o dataset](https://www.kaggle.com/datasets/krishujeniya/salary-prediction-of-data-professions?resource=download), são:
* FIRST NAME: Primeiro nome do profissional de dados (String)
* LAST NAME: Sobrenome do profissional de dados (String)
* SEX: Gênero do profissional de dados (String: 'F' para Feminino, 'M' para Masculino)
* DOJ (Date of Joining): A data em que o profissional de dados ingressou na empresa (Data no formato MM/DD/AAAA)
* CURRENT DATE: A data atual ou a data de referência dos dados (Data no formato MM/DD/AAAA)
* DESIGNATION: O cargo ou designação do profissional de dados (String: ex., Analista, Analista Sênior, Gerente)
* AGE: Idade do profissional de dados (Integer)
* SALARY: Salário anual do profissional de dados (Float)
* UNIT: Unidade de negócios ou departamento em que o profissional de dados trabalha (String: ex., TI, Finanças, Marketing)
* LEAVES USED: Número de licenças utilizadas pelo profissional de dados (Integer)
* LEAVES REMAINING: Número de licenças restantes para o profissional de dados (Integer)
* RATINGS: Avaliações de desempenho do profissional de dados (Float)
* PAST EXP: Experiência de trabalho anterior em anos antes de ingressar na empresa atual (Float)

Na célula a seguir, **carregue os dados do CSV e dê uma olhada neles antes de seguir**.

In [1]:
### Escreva sua resposta aqui
import pandas as pd

df = pd.read_csv("salaries.csv")
df.head()

,FIRST NAME,LAST NAME,SEX,DOJ,CURRENT DATE,DESIGNATION,AGE,SALARY,UNIT,LEAVES USED,LEAVES REMAINING,RATINGS,PAST EXP
0,TOMASA,ARMEN,F,5-18-2014,01-07-2016,Analyst,21.0,44570,Finance,24.0,6.0,2.0,0
1,ANNIE,NaN,F,NaN,01-07-2016,Associate,NaN,89207,Web,NaN,13.0,NaN,7
2,OLIVE,ANCY,F,7-28-2014,01-07-2016,Analyst,21.0,40955,Finance,23.0,7.0,3.0,0
3,CHERRY,AQUILAR,F,04-03-2013,01-07-2016,Analyst,22.0,45550,IT,22.0,8.0,3.0,0
4,LEON,ABOULAHOUD,M,11-20-2014,01-07-2016,Analyst,NaN,43161,Operations,27.0,3.0,NaN,3


#### Q2. Modelando os dados
Você deve **criar um ORM com SQLAlchemy capaz de comportar os dados dessa base**.

* Crie um campo de chave primária `ID`, que deve ser incrementado automaticamente
* Os campos SEX, DESIGNATION e UNIT devem ser definidos como classes `Enum` com os possíveis valores (consulte os valores únicos dessas colunas)
* Para os outros campos, consulte os tipos de dados informados na descrição acima

In [20]:
### Escreva sua resposta aqui
import enum
from sqlalchemy import Column, Integer, Float, String, Date, Enum
from sqlalchemy.orm import declarative_base

Base = declarative_base()

class Sexo(enum.Enum):
    F = "F"
    M = "M"

class Cargo(enum.Enum):
    Analyst = "Analyst"
    Associate = "Associate"
    Senior_Analyst = "Senior Analyst"
    Manager = "Manager"
    Senior_Manager = "Senior Manager"
    Director = "Director"

class Unidade(enum.Enum):
    Finance = "Finance"
    Web = "Web"
    IT = "IT"
    Operations = "Operations"
    Marketing = "Marketing"
    Management = "Management"

def _values(enum_cls):
    return [e.value for e in enum_cls]

class Salario(Base):
    __tablename__ = "salarios"

    id = Column(Integer, primary_key=True, autoincrement=True)
    first_name = Column(String)
    last_name = Column(String)
    sex = Column(Enum(Sexo, values_callable=_values))
    doj = Column(Date)
    current_date = Column(Date)
    designation = Column(Enum(Cargo, values_callable=_values))
    age = Column(Float)
    salary = Column(Float)
    unit = Column(Enum(Unidade, values_callable=_values))
    leaves_used = Column(Float)
    leaves_remaining = Column(Float)
    ratings = Column(Float)
    past_exp = Column(Float)

ModuleNotFoundError: No module named 'sqlalchemy'

#### Q3. Estabelecendo uma conexão

Usando o método `create_engine` do SQLAlchemy, crie uma conexão com um novo banco de dados SQLite chamado `salarios`.

In [ ]:
### Escreva sua resposta aqui
from sqlalchemy import create_engine

engine = create_engine("sqlite:///salarios.db")

#### Q4. Criando as tabelas
Crie as tabelas da questão Q2 no banco `salarios`.

In [ ]:
### Escreva sua resposta aqui
Base.metadata.create_all(engine)

#### Q5. Populando

Usando o método `to_sql` da biblioteca Pandas (veja [a documentação](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html)), popule o banco `salarios` com os dados do csv que você carregou na questão Q1.
* Lembre-se de definir o parâmetro `if_exists='append'` para que as tabelas não sejam dropadas e recriadas.

In [ ]:
### Escreva sua resposta aqui
# ajustar datas e nomes de colunas para bater com o modelo
df["DOJ"] = pd.to_datetime(df["DOJ"], format="%m-%d-%Y", errors="coerce")
df["CURRENT DATE"] = pd.to_datetime(df["CURRENT DATE"], format="%m-%d-%Y", errors="coerce")

df = df.rename(columns={
    "FIRST NAME": "first_name",
    "LAST NAME": "last_name",
    "SEX": "sex",
    "DOJ": "doj",
    "CURRENT DATE": "current_date",
    "DESIGNATION": "designation",
    "AGE": "age",
    "SALARY": "salary",
    "UNIT": "unit",
    "LEAVES USED": "leaves_used",
    "LEAVES REMAINING": "leaves_remaining",
    "RATINGS": "ratings",
    "PAST EXP": "past_exp",
})

df.to_sql("salarios", con=engine, if_exists="append", index=False)

#### Q6. Consultas SQL vs ORM

Agrupe os dados por DESIGNATION e selecione o mínimo, máximo e a média dos salários (SALARY) divididos por 12. Já que o atributo SALARY é anual, dividir por 12 nos mostrará os valores mensais.

Assumindo que a variável que armazena a sua conexão se chama `engine`, você deve realizar a query acima de três formas:
* Executando a query SQL através de uma instância de conexão retornada pelo método `engine.connect()`
* Executando a query SQL com o método `read_sql_query` do Pandas (veja [a documentação](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html)). Você usará mesma instância `engine.connect()` como um dos parâmetros.
* Executando uma query criada com o módulo `select` do SQLAlchemy. Sua execução deve ser feita através de um objeto `Session` do módulo `orm` do SQLAlchemy (`Session(engine)`).


In [ ]:
### Execute aqui sua query SQL com SQLAlchemy
from sqlalchemy import text, select, func
from sqlalchemy.orm import Session

sql = text("""
    SELECT designation,
           MIN(salary) / 12.0 AS salario_min_mensal,
           MAX(salary) / 12.0 AS salario_max_mensal,
           AVG(salary) / 12.0 AS salario_medio_mensal
    FROM salarios
    GROUP BY designation
""")

# (a) SQL puro
with engine.connect() as conn:
    resultado_sql = conn.execute(sql).fetchall()
print(resultado_sql)

In [ ]:
### Execute aqui sua query SQL com SQLAlchemy + Pandas
# (b) SQL + Pandas
with engine.connect() as conn:
    resultado_pandas = pd.read_sql_query(sql, con=conn)
resultado_pandas

In [ ]:
### Execute aqui sua query com SQLAlchemy ORM
# (c) ORM com select + Session
query_orm = select(
    Salario.designation,
    (func.min(Salario.salary) / 12.0).label("salario_min_mensal"),
    (func.max(Salario.salary) / 12.0).label("salario_max_mensal"),
    (func.avg(Salario.salary) / 12.0).label("salario_medio_mensal"),
).group_by(Salario.designation)

with Session(engine) as session:
    resultado_orm = session.execute(query_orm).all()
print(resultado_orm)